# SkinSight GPU training
Enable **GPU** and **Internet** in Kaggle settings, then Run All. This notebook contains the corrected app and training code; no Git push is needed. It downloads several GB of public datasets. Full-data/GPU execution was not run in the development workspace. Dataset source attribution and terms apply. Results are research metrics, not clinical accuracy. Do not repeatedly tune using the held-out test. Official ISIC metadata provides lesion groups where present. Lesion groups do not establish patient separation. Exact-file hashes also prevent identical files crossing splits. A fixed held-out test is evaluated only after validation selects the checkpoint.

In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
# Preserve Kaggle's matching CUDA torch/torchvision packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kagglehub==0.3.12',
    'fastapi==0.115.12', 'uvicorn==0.34.2', 'python-multipart==0.0.20',
    'jinja2==3.1.6', 'albumentations==2.0.7', 'pandas==2.2.3',
    'scikit-learn==1.6.1', 'opencv-python-headless==4.11.0.86'], check=True)
import torchvision
print('torch:',torch.__version__, 'torchvision:',torchvision.__version__)


In [ ]:
PROJECT = Path('/kaggle/working/SkinSight_AI')
PROJECT.mkdir(parents=True, exist_ok=True)
FILES = {'.gitignore': '.venv/\n__pycache__/\n*.pyc\ndata/raw/*\n!data/raw/.gitkeep\nmodels/*.pt\nmodels/*.pth\nmodels/*.onnx\n', 'README.md': '# SkinSight AI\n\nEducational skin-lesion classification using **MobileNetV3-Small**. Six classes:\nACK, BCC, MEL, NEV, SCC, SEK. ISIC AK maps to ACK, NV to NEV, BKL to SEK.\nBKL is broader than PAD\'s seborrheic keratosis, so this mapping is approximate.\n\n## Easiest GPU training workflow\n\nImport `notebooks/train_kaggle.ipynb` into Kaggle. Enable an available GPU and\nInternet, then Run All. The notebook bundles this code, downloads the public\nPAD and resized ISIC datasets plus ground truth, validates matching, trains,\nand exports `skinsight_trained_app.zip`. GPU/account access and full-data downloads\nhave not been verified here. Follow dataset licenses and attribution requirements.\n\n## Windows local workflow\n\nRun these commands from `SkinSight_AI` (Python 3.12 recommended):\n\n```powershell\npy -m venv .venv\n.\\.venv\\Scripts\\python -m pip install torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cpu\n.\\.venv\\Scripts\\python -m pip install -r requirements.txt\n.\\.venv\\Scripts\\python ml\\prepare_combined.py --pad-meta "D:\\SkinCancerData\\PAD-UFES-20\\extracted\\metadata.csv" --pad-root "D:\\SkinCancerData\\PAD-UFES-20\\extracted" --isic-gt "D:\\SkinCancerData\\ISIC-2019\\ISIC_2019_Training_GroundTruth.csv" --isic-root "D:\\SkinCancerData\\ISIC-2019\\extracted\\train-image" --out data\\combined_manifest.csv\n.\\.venv\\Scripts\\python ml\\train.py --manifest data\\combined_manifest.csv --epochs 10 --batch-size 16 --size 160 --fine-tune --device auto --output-dir models\n.\\.venv\\Scripts\\python run.py\n```\n\nPAD image folders are scanned recursively. Missing images cause an error instead\nof a partial manifest. CPU training on a low-memory laptop is likely slow.\n`--device auto` selects CUDA if available; `--device cuda` requires a CUDA GPU.\nPretrained weights download from PyTorch. Without `--fine-tune`, the backbone and\nits BatchNorm statistics stay frozen. `--from-scratch` requires `--fine-tune`.\n\nOutputs: `models/skinsight_mobilenetv3_small.pt`, `metrics.json`, `train_split.csv`,\n`validation_split.csv`. Reports include per-source validation metrics.\n\n## Connect trained weights\n\nCopy the trained checkpoint to `SkinSight_AI/models/skinsight_mobilenetv3_small.pt`\nand restart the app. Alternatively set `SKINSIGHT_MODEL_PATH` to an absolute path.\nOnly use trusted checkpoints; loading uses `weights_only=True`.\nWithout a checkpoint `/api/health` reports `model_ready=false` and predictions\nreturn HTTP 503. Synthetic test weights must never be deployed.\n\n## Evaluation limits\n\nPAD groups by real patient IDs. ISIC ground truth has no patient IDs; official\nmetadata supplies lesion groups where present. Lesion or image grouping **does not\nestablish patient-level separation**. The resized mirror\'s `dummy_*` IDs are not\ngenuine patient IDs. Validation selects the checkpoint; a separate internal test\npartition evaluates it afterward. Genuine ISIC patient metadata, probability\ncalibration, and external smartphone testing remain needed. Source/class balancing cannot\neliminate clinical-versus-dermoscopic domain shift. The 224px mirror is a practical\nbaseline; original ISIC images contain more detail.\n\nThis app is an educational research tool, not a diagnosis or clinical staging\nsystem. No real-data model has yet been trained or clinically validated here.\n\n## Tests\n\n```powershell\n.\\.venv\\Scripts\\python -m unittest discover -s tests -v\n```\n\nRegression tests use artificial images to check software behavior only.\n\n## Independent holdout and grouping\n\nTraining now creates a fixed, source/class-stratified **train/validation/test**\npartition (approximately 60/20/20). Linked patient IDs, lesion IDs, image IDs,\nand exact-file SHA-256 duplicates stay in a single partition. Each partition must\ncontain every source/class represented in the manifest; otherwise training fails\nwith a diagnostic rather than silently reporting incomplete results.\n\nSupply `--isic-meta path/to/ISIC_2019_Training_Metadata.csv` to preparation to\nuse authoritative ISIC lesion IDs and genuine patient IDs where available.\nThe Kaggle notebook downloads the official metadata automatically. Dummy patient\nIDs are ignored. `split_summary.json` records grouping coverage; missing IDs remain\nexplicit image-level groups. File hashes detect exact duplicates, not re-encoded\nor cropped copies. Lesion separation still does not prove patient separation.\n\nThe test partition is never used to select checkpoints. After training, the best\nvalidation checkpoint is loaded and test metrics are computed once. Outputs now\nalso include `test_split.csv`, `split_summary.json`, and `test_metrics.json`.\nOverall and per-source test reports contain confusion matrices, per-class\nsensitivity/specificity, one-vs-rest ROC AUC (null when undefined), and calibration\nerror. PAD-only test metrics are the relevant internal smartphone-photo measure.\nExternal smartphone performance remains unknown until separately evaluated.\n\nUse a fresh output directory for another run. Reusing test feedback to tune a\nmodel contaminates the test set even when filenames are different. Existing\ncheckpoints/test results will not be overwritten automatically.\n', 'requirements.txt': 'fastapi==0.115.12\nuvicorn[standard]==0.34.2\npython-multipart==0.0.20\njinja2==3.1.6\npillow==11.2.1\nnumpy==2.2.5\ntorch==2.7.0\ntorchvision==0.22.0\ntimm==1.0.15\npandas==2.2.3\nscikit-learn==1.6.1\nalbumentations==2.0.7\nopencv-python-headless==4.11.0.86\n', 'run.py': 'import argparse\nimport uvicorn\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--host", default="127.0.0.1")\n    parser.add_argument("--port", type=int, default=8000)\n    parser.add_argument("--reload", action="store_true")\n    args = parser.parse_args()\n    uvicorn.run("app.main:app", host=args.host, port=args.port, reload=args.reload)\n', 'DEVELOPMENT_STATUS.md': '# Verified development status\n\nThe app and trainer now share torchvision MobileNetV3-Small and the same checkpoint\nfilename. The predictor uses a restricted checkpoint loader. Static assets and the\ndefault checkpoint resolve relative to the application files rather than the shell.\n\nDataset preparation validates one-hot ISIC labels and image matching, preserves\npatient/lesion grouping where metadata exists, and hashes image files. Training\nreserves train/validation/test folds, checks source/class coverage, selects weights\non validation only, and exports overall and source-specific held-out reports.\n\n## Verification completed\n\n- Twelve regression tests: HTTP prediction, invalid image and missing model responses,\n  checkpoint loading, recursive manifests, one-hot validation, dummy-ID handling,\n  transitive identity grouping, partition isolation, and metric calculations.\n- Two epochs on 240 synthetic images: training, best-checkpoint selection, test\n  isolation, report generation, checkpoint reload, and HTTP prediction.\n- Uploaded metadata: 2,298 PAD records and 24,839 supported ISIC records permit\n  source/class coverage in three partitions. This metadata-only feasibility check\n  did not include image hashes or the official ISIC lesion metadata; final folds\n  must be generated from actual downloaded images and metadata.\n- Dependencies passed pip check. Notebook code cells passed syntax compilation;\n  exported source ZIP passed integrity checks and contains no synthetic checkpoint.\n\n## Outstanding external work\n\nNo real-image model has been trained in this workspace. Kaggle and official ISIC\nHTTP requests were denied by the cloud egress proxy. GPU hardware and Kaggle runtime\naccess are unavailable here. Network domain additions were saved to the environment\ndraft but were not applied to the live instance during validation.\n\nFull downloads and GPU notebook execution remain unverified. Research accuracy,\nexternal smartphone performance, genuine ISIC patient separation, and probability\ncalibration cannot be inferred from software tests. The notebook includes a fixed\ninternal test; repeated tuning against its results invalidates independence.\n\nThe next supported step is to apply the saved network settings and retry cloud\naccess, or execute the bundled notebook in a Kaggle account with GPU/Internet access.\nOnly deploy the resulting real-data checkpoint after reviewing the test report.\n', 'app/guidance.py': 'GUIDANCE = {\n    "ACK": {\n        "name": "Actinic Keratosis",\n        "risk": "Needs dermatologist review",\n        "message": "Often related to sun damage and can be precancerous. Arrange a dermatologist review.",\n        "urgency": "Book a routine-to-soon dermatology appointment, especially if changing, tender, crusting, or bleeding."\n    },\n    "BCC": {\n        "name": "Basal Cell Carcinoma",\n        "risk": "High concern",\n        "message": "This class represents a skin cancer category in the training dataset.",\n        "urgency": "Arrange prompt in-person dermatology assessment and biopsy consideration."\n    },\n    "MEL": {\n        "name": "Melanoma",\n        "risk": "High concern",\n        "message": "This class represents melanoma, which requires professional assessment.",\n        "urgency": "Arrange prompt dermatologist evaluation. Do not rely on this image result to confirm or rule out melanoma."\n    },\n    "NEV": {\n        "name": "Melanocytic Nevus",\n        "risk": "Usually lower concern",\n        "message": "Many nevi are benign, but changing or unusual lesions still need examination.",\n        "urgency": "Seek review if the lesion is new, changing, asymmetric, bleeding, itching, painful, or looks different from your other moles."\n    },\n    "SCC": {\n        "name": "Squamous Cell Carcinoma",\n        "risk": "High concern",\n        "message": "This class represents a skin cancer category in the training dataset.",\n        "urgency": "Arrange prompt in-person dermatology assessment and biopsy consideration."\n    },\n    "SEK": {\n        "name": "Seborrheic Keratosis",\n        "risk": "Usually lower concern",\n        "message": "This is commonly benign, but visual similarity with other lesions can occur.",\n        "urgency": "Seek clinical review if rapidly changing, bleeding, painful, or uncertain."\n    },\n}\n\ndef stage_info(label: str):\n    if label in {"BCC", "MEL", "SCC"}:\n        return {\n            "status": "Not determinable from a photo",\n            "explanation": (\n                "Clinical cancer Stage I–IV cannot be assigned safely from this image alone. "\n                "Staging requires pathology/biopsy and may require tumor depth, ulceration, "\n                "lymph-node findings, imaging, or evidence of spread."\n            ),\n        }\n    return {\n        "status": "Not applicable from image classification",\n        "explanation": (\n            "The image model provides lesion-class screening, not clinical cancer staging."\n        ),\n    }\n', 'app/inference.py': 'from pathlib import Path\nimport io\nimport os\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torchvision.models import mobilenet_v3_small\nfrom torch import nn\nimport albumentations as A\nfrom albumentations.pytorch import ToTensorV2\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nCHECKPOINT = Path(os.environ.get("SKINSIGHT_MODEL_PATH", str(PROJECT_ROOT / "models/skinsight_mobilenetv3_small.pt")))\n\nclass Predictor:\n    def __init__(self):\n        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        self.model = None\n        self.class_names = None\n        self.image_size = 160\n        self.meta = {}\n        self._load()\n\n    def _load(self):\n        if not CHECKPOINT.exists():\n            return\n        ckpt = torch.load(CHECKPOINT, map_location=self.device, weights_only=True)\n        self.class_names = ckpt["class_names"]\n        self.image_size = int(ckpt.get("image_size", 160))\n        self.meta = {\n            "architecture": ckpt.get("architecture", "mobilenet_v3_small"),\n            "dataset": ckpt.get("dataset", "PAD-UFES-20"),\n            "balanced_accuracy": ckpt.get("balanced_accuracy"),\n            "training_mode": ckpt.get("training_mode"),\n        }\n        if self.meta["architecture"] != "mobilenet_v3_small":\n            raise ValueError("Expected a MobileNetV3-Small checkpoint produced by ml/train.py")\n        if set(self.class_names) != {"ACK", "BCC", "MEL", "NEV", "SCC", "SEK"} or len(self.class_names) != 6:\n            raise ValueError("Checkpoint must contain the six supported lesion classes")\n        self.model = mobilenet_v3_small(weights=None)\n        self.model.classifier[-1] = nn.Linear(\n            self.model.classifier[-1].in_features, len(self.class_names)\n        )\n        self.model.load_state_dict(ckpt["model_state"])\n        self.model.to(self.device).eval()\n\n        self.transform = A.Compose([\n            A.Resize(self.image_size, self.image_size),\n            A.Normalize(),\n            ToTensorV2(),\n        ])\n\n    @property\n    def ready(self):\n        return self.model is not None\n\n    @torch.no_grad()\n    def predict_bytes(self, raw: bytes):\n        if not self.ready:\n            raise RuntimeError("Model is not trained yet. Train it first with ml/train.py.")\n\n        image = Image.open(io.BytesIO(raw)).convert("RGB")\n        arr = np.asarray(image)\n        x = self.transform(image=arr)["image"].unsqueeze(0).to(self.device)\n        logits = self.model(x)\n        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()\n\n        order = np.argsort(probs)[::-1]\n        ranked = [\n            {"label": self.class_names[i], "probability": round(float(probs[i]) * 100, 2)}\n            for i in order[:3]\n        ]\n        return {\n            "label": ranked[0]["label"],\n            "confidence": ranked[0]["probability"],\n            "top_predictions": ranked,\n        }\n', 'app/main.py': 'from fastapi import FastAPI, File, HTTPException, Request, UploadFile\nfrom fastapi.responses import HTMLResponse, JSONResponse\nfrom fastapi.staticfiles import StaticFiles\nfrom fastapi.templating import Jinja2Templates\nfrom PIL import Image\nimport io\nfrom pathlib import Path\n\nfrom .inference import Predictor\nfrom .guidance import GUIDANCE, stage_info\n\napp = FastAPI(\n    title="SkinSight AI",\n    description="Educational skin-lesion screening using an own-trained MobileNetV3-Small model.",\n    version="0.1.0",\n)\n\nAPP_DIR = Path(__file__).resolve().parent\napp.mount("/static", StaticFiles(directory=str(APP_DIR / "static")), name="static")\ntemplates = Jinja2Templates(directory=str(APP_DIR / "templates"))\npredictor = Predictor()\n\n@app.get("/", response_class=HTMLResponse)\ndef home(request: Request):\n    return templates.TemplateResponse(\n        "index.html",\n        {"request": request, "model_ready": predictor.ready, "model_meta": predictor.meta},\n    )\n\n@app.get("/api/health")\ndef health():\n    return {\n        "status": "ok",\n        "model_ready": predictor.ready,\n        "model": predictor.meta if predictor.ready else None,\n    }\n\n@app.post("/api/predict")\nasync def predict(image: UploadFile = File(...)):\n    if not predictor.ready:\n        raise HTTPException(\n            status_code=503,\n            detail="The model has not been trained yet. Run ml/train.py first.",\n        )\n\n    if image.content_type not in {"image/jpeg", "image/png", "image/webp"}:\n        raise HTTPException(status_code=400, detail="Please upload a JPG, PNG, or WEBP image.")\n\n    raw = await image.read()\n    if len(raw) > 10 * 1024 * 1024:\n        raise HTTPException(status_code=413, detail="Image must be 10 MB or smaller.")\n\n    try:\n        im = Image.open(io.BytesIO(raw))\n        im.verify()\n    except Exception:\n        raise HTTPException(status_code=400, detail="The uploaded file is not a valid image.")\n\n    try:\n        result = predictor.predict_bytes(raw)\n    except Exception as exc:\n        raise HTTPException(status_code=500, detail=str(exc))\n\n    label = result["label"]\n    result["guidance"] = GUIDANCE[label]\n    result["stage"] = stage_info(label)\n    result["medical_disclaimer"] = (\n        "This is an educational AI screening result, not a medical diagnosis. "\n        "Model scores are not calibrated probabilities of disease. A clinician must assess concerning lesions."\n    )\n    return JSONResponse(result)\n', 'ml/dataset.py': 'from pathlib import Path\nimport pandas as pd\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nCLASS_NAMES = ["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"]\nCLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}\n\nclass ManifestDataset(Dataset):\n    """\n    Dataset backed by data/combined_manifest.csv.\n\n    Required columns:\n      image_path,label,patient_id,source,original_id\n    """\n    def __init__(self, frame, transform=None):\n        self.df = frame.reset_index(drop=True)\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, idx):\n        row = self.df.iloc[idx]\n        path = Path(row["image_path"])\n        image = Image.open(path).convert("RGB")\n        label = str(row["label"]).strip().upper()\n        if label not in CLASS_TO_IDX:\n            raise ValueError(f"Unsupported label: {label}")\n        y = CLASS_TO_IDX[label]\n\n        if self.transform:\n            import numpy as np\n            image = self.transform(image=np.asarray(image))["image"]\n\n        return image, y\n', 'ml/model.py': 'import torch.nn as nn\nfrom torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights\n\nNUM_CLASSES = 6\nCLASS_NAMES = ["ACK","BCC","MEL","NEV","SCC","SEK"]\n\ndef create_model(num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True):\n    weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None\n    model = mobilenet_v3_small(weights=weights)\n    if freeze_backbone:\n        for p in model.features.parameters():\n            p.requires_grad = False\n    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)\n    return model\n', 'ml/prepare_combined.py': 'import argparse\nimport hashlib\nfrom pathlib import Path\nimport pandas as pd\n\nTARGET_CLASSES = ["ACK", "BCC", "MEL", "NEV", "SCC", "SEK"]\n\nPAD_ALIASES = {\n    "ACK": "ACK",\n    "AK": "ACK",\n    "ACTINIC KERATOSIS": "ACK",\n    "BCC": "BCC",\n    "BASAL CELL CARCINOMA": "BCC",\n    "MEL": "MEL",\n    "MELANOMA": "MEL",\n    "NEV": "NEV",\n    "NEVUS": "NEV",\n    "SCC": "SCC",\n    "SQUAMOUS CELL CARCINOMA": "SCC",\n    "SEK": "SEK",\n    "SK": "SEK",\n    "SEBORRHEIC KERATOSIS": "SEK",\n}\n\nISIC_MAP = {\n    "MEL": "MEL",\n    "NV": "NEV",\n    "BCC": "BCC",\n    "AK": "ACK",\n    "BKL": "SEK",\n    "SCC": "SCC",\n}\n\nMANIFEST_COLUMNS = ["image_path", "label", "patient_id", "lesion_id", "grouping_level", "image_sha256", "source", "original_id"]\n\ndef digest(path):\n    with path.open("rb") as file:\n        return hashlib.file_digest(file, "sha256").hexdigest()\n\nVALID_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}\n\ndef find_column(df, candidates, required=True):\n    lookup = {str(c).strip().lower(): c for c in df.columns}\n    for c in candidates:\n        if c.lower() in lookup:\n            return lookup[c.lower()]\n    if required:\n        raise ValueError(f"Could not find any of {candidates}. Found columns: {list(df.columns)}")\n    return None\n\ndef build_image_index(root: Path):\n    if not root.is_dir():\n        raise FileNotFoundError(f"Image directory does not exist: {root}")\n    index = {}\n    for p in root.rglob("*"):\n        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:\n            if p.stem in index and index[p.stem] != p.resolve():\n                raise ValueError(f"Ambiguous image ID: {p.stem}")\n            index.setdefault(p.stem, p.resolve())\n            index.setdefault(p.name, p.resolve())\n    return index\n\ndef normalize_pad(v):\n    return PAD_ALIASES.get(str(v).strip().upper())\n\ndef build_pad_manifest(metadata_csv: Path, pad_root: Path):\n    df = pd.read_csv(metadata_csv)\n    image_col = find_column(df, ["img_id", "image", "image_id", "filename", "file_name"])\n    label_col = find_column(df, ["diagnostic", "diagnosis", "label", "class"])\n    patient_col = find_column(df, ["patient_id", "patient", "patientid"], required=False)\n\n    lesion_col = find_column(df, ["lesion_id"], required=False)\n    if df[image_col].duplicated().any():\n        raise ValueError("PAD metadata has duplicate image IDs")\n    image_index = build_image_index(pad_root)\n    rows = []\n    missing = 0\n\n    for _, row in df.iterrows():\n        label = normalize_pad(row[label_col])\n        if label not in TARGET_CLASSES:\n            continue\n\n        raw = str(row[image_col]).strip()\n        path = image_index.get(raw) or image_index.get(Path(raw).stem)\n\n        if path is None:\n            missing += 1\n            continue\n\n        patient_id = ""\n        if patient_col and pd.notna(row[patient_col]):\n            patient_id = f"PAD::{row[patient_col]}"\n\n        rows.append({\n            "image_path": str(path),\n            "label": label,\n            "patient_id": patient_id,\n            "lesion_id": str(row[lesion_col]) if lesion_col and pd.notna(row[lesion_col]) else "",\n            "grouping_level": "patient" if patient_id else ("lesion" if lesion_col and pd.notna(row[lesion_col]) else "image"),\n            "image_sha256": digest(path),\n            "source": "PAD-UFES-20",\n            "original_id": Path(raw).stem,\n        })\n\n    if missing:\n        raise ValueError(f"{missing} supported images are missing; check the image directory")\n    out = pd.DataFrame(rows, columns=MANIFEST_COLUMNS)\n    if out.empty:\n        raise ValueError("No supported images matched the metadata")\n    print(f"PAD matched images: {len(out)} | missing: {missing}")\n    return out\n\ndef build_isic_manifest(gt_csv: Path, image_root: Path, metadata_csv=None):\n    gt = pd.read_csv(gt_csv)\n    image_col = find_column(gt, ["image", "image_id", "isic_id"])\n    if gt[image_col].duplicated().any():\n        raise ValueError("ISIC ground truth has duplicate image IDs")\n    label_columns = [name for name in [*ISIC_MAP, "DF", "VASC", "UNK"] if name in gt]\n    if not label_columns:\n        raise ValueError("ISIC ground truth has no recognized class columns")\n    numeric = gt[label_columns].apply(pd.to_numeric, errors="raise")\n    if not numeric.isin([0, 1]).all().all() or not numeric.sum(axis=1).eq(1).all():\n        raise ValueError("ISIC labels must be valid one-hot rows")\n    metadata = {}\n    if metadata_csv:\n        meta = pd.read_csv(metadata_csv)\n        meta_image = find_column(meta, ["image", "image_id", "isic_id"])\n        if meta[meta_image].duplicated().any():\n            raise ValueError("ISIC metadata has duplicate image IDs")\n        metadata = {str(row[meta_image]): row for _, row in meta.iterrows()}\n    image_index = build_image_index(image_root)\n\n    rows = []\n    missing = 0\n    skipped = 0\n\n    for _, row in gt.iterrows():\n        positives = []\n        for src_label in ISIC_MAP:\n            if src_label in gt.columns:\n                try:\n                    if float(row[src_label]) >= 0.5:\n                        positives.append(src_label)\n                except Exception:\n                    pass\n\n        if len(positives) != 1:\n            skipped += 1\n            continue\n\n        src_label = positives[0]\n        final_label = ISIC_MAP[src_label]\n        image_id = str(row[image_col]).strip()\n        path = image_index.get(image_id) or image_index.get(Path(image_id).stem)\n\n        if path is None:\n            missing += 1\n            continue\n\n        meta = metadata.get(image_id, {})\n        patient = meta.get("patient_id", "")\n        patient = str(patient).strip() if pd.notna(patient) else ""\n        if patient.lower().startswith("dummy_"):\n            patient = ""\n        lesion = meta.get("lesion_id", "")\n        lesion = str(lesion).strip() if pd.notna(lesion) else ""\n        rows.append({\n            "image_path": str(path),\n            "label": final_label,\n            "patient_id": patient,\n            "lesion_id": lesion,\n            "grouping_level": "patient" if patient else ("lesion" if lesion else "image"),\n            "image_sha256": digest(path),\n            "source": "ISIC-2019",\n            "original_id": Path(image_id).stem,\n        })\n\n    if missing:\n        raise ValueError(f"{missing} supported images are missing; check the image directory")\n    out = pd.DataFrame(rows, columns=MANIFEST_COLUMNS)\n    if out.empty:\n        raise ValueError("No supported images matched the metadata")\n    print(f"ISIC matched images: {len(out)} | missing: {missing} | unsupported/skipped: {skipped}")\n    return out\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--pad-meta", required=True)\n    p.add_argument("--pad-root", required=True)\n    p.add_argument("--isic-gt", required=True)\n    p.add_argument("--isic-root", required=True)\n    p.add_argument("--isic-meta", help="Optional authoritative ISIC metadata with lesion/patient identifiers")\n    p.add_argument("--out", default="data/combined_manifest.csv")\n    args = p.parse_args()\n\n    pad = build_pad_manifest(Path(args.pad_meta), Path(args.pad_root))\n    isic = build_isic_manifest(Path(args.isic_gt), Path(args.isic_root), Path(args.isic_meta) if args.isic_meta else None)\n\n    combined = pd.concat([pad, isic], ignore_index=True)\n    combined = combined.drop_duplicates(subset=["source", "original_id"]).reset_index(drop=True)\n\n    print("\\nGrouping coverage:"); print(combined.groupby(["source", "grouping_level"]).size())\n    if (combined.grouping_level == "image").any():\n        print("WARNING: image-only groups cannot guarantee patient/lesion separation.")\n    out = Path(args.out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    combined.to_csv(out, index=False)\n\n    print("\\n=== FINAL COMBINED DATASET ===")\n    print(combined.groupby(["source", "label"]).size().unstack(fill_value=0))\n    print("\\nClass totals:")\n    print(combined["label"].value_counts().reindex(TARGET_CLASSES, fill_value=0))\n    print("\\nGrand total:", len(combined))\n    print("\\nSaved manifest:", out)\n\nif __name__ == "__main__":\n    main()\n', 'ml/train.py': '"""Train on training data, select on validation, evaluate held-out test once."""\nimport argparse\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\nimport albumentations as A\nfrom albumentations.pytorch import ToTensorV2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, WeightedRandomSampler\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom dataset import ManifestDataset, CLASS_NAMES\nfrom model import create_model\nfrom splits import linked_groups, split_three_way\nfrom evaluation import summarize\n\n\ndef seed_all(seed=42):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n\n\ndef tfms(train, size):\n    items = [A.Resize(size, size)]\n    if train:\n        items += [A.HorizontalFlip(p=.5), A.Rotate(limit=20, p=.3),\n                  A.RandomBrightnessContrast(p=.25), A.HueSaturationValue(p=.15)]\n    return A.Compose(items + [A.Normalize(), ToTensorV2()])\n\n\ndef split_df(df):\n    """Compatibility helper for callers needing only a train/validation partition."""\n    df = df[df.label.isin(CLASS_NAMES)].copy().reset_index(drop=True)\n    if df.empty:\n        raise ValueError(\'Manifest contains no supported lesion classes\')\n    df[\'_group\'] = linked_groups(df)\n    a, b = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42).split(df, groups=df[\'_group\']))\n    return df.iloc[a].copy(), df.iloc[b].copy()\n\n\ndef sampler(df):\n    keys = df.source.astype(str) + \'::\' + df.label.astype(str)\n    counts = keys.value_counts().to_dict()\n    return WeightedRandomSampler([1 / counts[key] for key in keys], len(keys), replacement=True)\n\n\n@torch.no_grad()\ndef collect(model, loader, device):\n    model.eval(); labels = []; probabilities = []; loss_sum = 0.\n    for x, y in loader:\n        x, y = x.to(device), y.to(device)\n        logits = model(x)\n        loss_sum += float(nn.functional.cross_entropy(logits, y)) * len(y)\n        labels.extend(y.cpu().tolist())\n        probabilities.extend(torch.softmax(logits, 1).cpu().tolist())\n    return loss_sum / len(loader.dataset), labels, probabilities\n\n\ndef evaluate(model, loader, device):\n    loss, labels, probabilities = collect(model, loader, device)\n    report = summarize(labels, probabilities)\n    return loss, report[\'balanced_accuracy\'], report[\'classification_report\']\n\n\ndef report_frame(model, frame, size, batch_size, workers, device):\n    loader = DataLoader(ManifestDataset(frame, tfms(False, size)), batch_size=batch_size,\n                        shuffle=False, num_workers=workers)\n    loss, labels, probabilities = collect(model, loader, device)\n    report = summarize(labels, probabilities); report[\'loss\'] = loss\n    report[\'by_source\'] = {}\n    sources = frame.source.to_numpy()\n    for source in sorted(frame.source.unique()):\n        selected = np.flatnonzero(sources == source)\n        report[\'by_source\'][source] = summarize(np.asarray(labels)[selected], np.asarray(probabilities)[selected])\n    return report\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--manifest\', default=\'data/combined_manifest.csv\')\n    p.add_argument(\'--epochs\', type=int, default=10)\n    p.add_argument(\'--batch-size\', type=int, default=16)\n    p.add_argument(\'--workers\', type=int, default=0)\n    p.add_argument(\'--size\', type=int, default=160)\n    p.add_argument(\'--lr\', type=float, default=1e-3)\n    p.add_argument(\'--output-dir\', default=\'models\')\n    p.add_argument(\'--from-scratch\', action=\'store_true\')\n    p.add_argument(\'--fine-tune\', action=\'store_true\')\n    p.add_argument(\'--device\', choices=[\'auto\', \'cpu\', \'cuda\'], default=\'auto\')\n    a = p.parse_args(); seed_all()\n    if a.epochs < 1 or a.batch_size < 2 or a.size < 32 or a.workers < 0 or a.lr <= 0:\n        p.error(\'epochs/lr must be positive, batch-size >= 2, size >= 32, workers >= 0\')\n    if a.from_scratch and not a.fine_tune:\n        p.error(\'--from-scratch requires --fine-tune so random features are not frozen\')\n    if a.device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA requested, but no CUDA GPU is available\')\n    device = torch.device(\'cuda\' if a.device == \'auto\' and torch.cuda.is_available() else\n                          (\'cpu\' if a.device == \'auto\' else a.device))\n    out = Path(a.output_dir); checkpoint = out / \'skinsight_mobilenetv3_small.pt\'\n    if checkpoint.exists() or (out / \'test_metrics.json\').exists():\n        raise FileExistsError(\'Training output already exists; use a new --output-dir to preserve prior results\')\n    df = pd.read_csv(a.manifest)\n    required = {\'image_path\', \'label\', \'source\', \'original_id\'}\n    if required - set(df.columns):\n        raise ValueError(f\'Manifest lacks columns: {sorted(required-set(df.columns))}\')\n    if not set(df.label).issubset(CLASS_NAMES):\n        raise ValueError(\'Manifest contains unsupported labels\')\n    if \'image_sha256\' not in df or df.image_sha256.isna().any():\n        raise ValueError(\'Rebuild the manifest to include image_sha256 for duplicate-aware splitting\')\n    tr, va, test = split_three_way(df)\n    out.mkdir(parents=True, exist_ok=True)\n    for name, part in [(\'train\', tr), (\'validation\', va), (\'test\', test)]:\n        part.to_csv(out / f\'{name}_split.csv\', index=False)\n        print(name, len(part)); print(part.groupby([\'source\', \'label\']).size().unstack(fill_value=0))\n    grouping = df.groupby([\'source\', \'grouping_level\']).size().to_dict() if \'grouping_level\' in df else {}\n    split_info = {\n        \'seed\': 42, \'manifest_sha256\': hashlib.sha256(Path(a.manifest).read_bytes()).hexdigest(),\n        \'counts\': {\'train\': len(tr), \'validation\': len(va), \'test\': len(test)},\n        \'grouping_coverage\': {f\'{source}::{level}\': int(count) for (source, level), count in grouping.items()},\n        \'limitations\': \'Linked patient, lesion and exact-file hashes stay together. Image/lesion groups do not prove patient separation. Re-encoded near-duplicates and unseen external data need further evaluation.\',\n    }\n    (out / \'split_summary.json\').write_text(json.dumps(split_info, indent=2))\n    training = DataLoader(ManifestDataset(tr, tfms(True, a.size)), batch_size=a.batch_size,\n                          sampler=sampler(tr), num_workers=a.workers, pin_memory=device.type == \'cuda\',\n                          drop_last=len(tr) % a.batch_size == 1)\n    validation = DataLoader(ManifestDataset(va, tfms(False, a.size)), batch_size=a.batch_size,\n                            shuffle=False, num_workers=a.workers)\n    model = create_model(pretrained=not a.from_scratch, freeze_backbone=not a.fine_tune).to(device)\n    optimizer = torch.optim.AdamW([x for x in model.parameters() if x.requires_grad], lr=a.lr, weight_decay=1e-4)\n    criterion = nn.CrossEntropyLoss(label_smoothing=.05)\n    best = -1.; history = []; best_epoch = None\n    print(\'Device:\', device, \'Architecture: mobilenet_v3_small\', flush=True)\n    for epoch in range(1, a.epochs + 1):\n        start = time.time(); model.train(); run = 0.\n        if not a.fine_tune:\n            model.features.eval()\n        for i, (x, y) in enumerate(training, 1):\n            x, y = x.to(device), y.to(device); optimizer.zero_grad(set_to_none=True)\n            logits = model(x); loss = criterion(logits, y); loss.backward(); optimizer.step()\n            run += float(loss.detach()) * len(y)\n            if i % 100 == 0 or i == len(training):\n                print(f\'epoch {epoch}/{a.epochs}, batch {i}/{len(training)}\', flush=True)\n        vl, score, report = evaluate(model, validation, device)\n        history.append({\'epoch\': epoch, \'train_loss\': run / len(tr), \'validation_loss\': vl,\n                        \'validation_balanced_accuracy\': score, \'seconds\': time.time() - start})\n        print(history[-1], flush=True)\n        if score > best:\n            best = score; best_epoch = epoch\n            torch.save({\'model_state\': model.state_dict(), \'class_names\': CLASS_NAMES,\n                        \'architecture\': \'mobilenet_v3_small\', \'image_size\': a.size,\n                        \'balanced_accuracy\': best, \'validation_report\': report,\n                        \'dataset\': \'PAD-UFES-20 + ISIC-2019\',\n                        \'training_mode\': \'from_scratch\' if a.from_scratch else (\'imagenet_finetune\' if a.fine_tune else \'imagenet_head_only\')}, checkpoint)\n    # Only after selection, reload the best checkpoint and evaluate test data once.\n    best_checkpoint = torch.load(checkpoint, map_location=device, weights_only=True)\n    model.load_state_dict(best_checkpoint[\'model_state\'])\n    validation_report = report_frame(model, va, a.size, a.batch_size, a.workers, device)\n    test_report = report_frame(model, test, a.size, a.batch_size, a.workers, device)\n    test_report[\'evaluation_note\'] = \'Held out from fitting and checkpoint selection. Reusing this test set to tune the model invalidates independent-test claims. Patient independence depends on grouping coverage.\'\n    test_report[\'split_summary\'] = split_info\n    (out / \'test_metrics.json\').write_text(json.dumps(test_report, indent=2))\n    (out / \'metrics.json\').write_text(json.dumps({\'best_epoch\': best_epoch, \'best_balanced_accuracy\': best,\n                                                \'history\': history, \'validation\': validation_report}, indent=2))\n    print(\'Saved:\', checkpoint, \'and test_metrics.json\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'ml/evaluation.py': '"""Fixed-label metrics including sensitivity, specificity, and calibration error."""\nimport numpy as np\nfrom sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score\nfrom dataset import CLASS_NAMES\n\n\ndef summarize(labels, probabilities):\n    labels = np.asarray(labels, dtype=int)\n    probabilities = np.asarray(probabilities, dtype=float)\n    if not len(labels):\n        raise ValueError(\'Cannot evaluate an empty dataset\')\n    predictions = probabilities.argmax(axis=1)\n    matrix = confusion_matrix(labels, predictions, labels=range(len(CLASS_NAMES)))\n    total = int(matrix.sum())\n    per_class = {}\n    for i, name in enumerate(CLASS_NAMES):\n        tp = int(matrix[i, i]); fn = int(matrix[i].sum()) - tp\n        fp = int(matrix[:, i].sum()) - tp; tn = total - tp - fn - fp\n        binary = labels == i\n        per_class[name] = {\n            \'support\': tp + fn,\n            \'sensitivity\': tp / (tp + fn) if tp + fn else None,\n            \'specificity\': tn / (tn + fp) if tn + fp else None,\n            \'roc_auc\': float(roc_auc_score(binary, probabilities[:, i])) if binary.any() and (~binary).any() else None,\n        }\n    confidence = probabilities.max(axis=1)\n    correctness = predictions == labels\n    ece = 0.0\n    for low, high in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]):\n        mask = (confidence >= low) & ((confidence < high) if high < 1 else (confidence <= high))\n        if mask.any():\n            ece += float(mask.mean() * abs(correctness[mask].mean() - confidence[mask].mean()))\n    return {\n        \'sample_count\': len(labels),\n        \'balanced_accuracy\': float(balanced_accuracy_score(labels, predictions)),\n        \'classification_report\': classification_report(labels, predictions, labels=range(len(CLASS_NAMES)),\n                                                      target_names=CLASS_NAMES, zero_division=0, output_dict=True),\n        \'confusion_matrix\': matrix.tolist(), \'class_order\': CLASS_NAMES,\n        \'per_class\': per_class, \'expected_calibration_error_10_bins\': ece,\n        \'calibration_note\': \'Softmax probabilities are not clinically calibrated confidence estimates.\',\n    }\n', 'ml/splits.py': '"""Deterministic source/class-stratified splits with linked identity groups."""\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\n\n\ndef present(value):\n    return pd.notna(value) and str(value).strip().lower() not in {\'\', \'nan\', \'none\'}\n\n\ndef linked_groups(frame):\n    """Union patient/lesion/image identities and exact content duplicates transitively."""\n    parents = list(range(len(frame)))\n    def find(i):\n        while parents[i] != i:\n            parents[i] = parents[parents[i]]\n            i = parents[i]\n        return i\n    seen = {}\n    for i, row in enumerate(frame.to_dict(\'records\')):\n        source = str(row[\'source\'])\n        keys = [(\'image\', source, str(row[\'original_id\']))]\n        for column in [\'patient_id\', \'lesion_id\']:\n            value = row.get(column)\n            if present(value):\n                keys.append((column, source, str(value).strip()))\n        if present(row.get(\'image_sha256\')):\n            keys.append((\'sha256\', str(row[\'image_sha256\'])))\n        for key in keys:\n            if key in seen:\n                parents[find(i)] = find(seen[key])\n            else:\n                seen[key] = i\n    return pd.Series([f\'group::{find(i)}\' for i in range(len(frame))], index=frame.index)\n\n\ndef split_three_way(frame):\n    """Approx. 60/20/20; fixed folds, no search using model scores or test outcomes."""\n    frame = frame.copy().reset_index(drop=True)\n    frame[\'_group\'] = linked_groups(frame)\n    strata = frame.source.astype(str) + \'::\' + frame.label.astype(str)\n    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)\n    folds = list(splitter.split(frame, strata, groups=frame[\'_group\']))\n    validation_ids = folds[0][1]\n    test_ids = folds[1][1]\n    train_mask = ~frame.index.isin(list(validation_ids) + list(test_ids))\n    train = frame.loc[train_mask].copy()\n    validation = frame.iloc[validation_ids].copy()\n    test = frame.iloc[test_ids].copy()\n    parts = {\'train\': train, \'validation\': validation, \'test\': test}\n    groups = [set(part[\'_group\']) for part in parts.values()]\n    if any(groups[i] & groups[j] for i in range(3) for j in range(i)):\n        raise RuntimeError(\'Linked identity group leakage detected\')\n    # Fail instead of silently evaluating a source/class missing from a split.\n    required = set(strata)\n    for name, part in parts.items():\n        actual = set(part.source.astype(str) + \'::\' + part.label.astype(str))\n        if required - actual:\n            raise ValueError(f\'{name} lacks source/class groups: {sorted(required-actual)}. \'\n                             \'More independent groups or an explicitly designed split are needed.\')\n    return train, validation, test\n', 'tests/test_api.py': '"""Run the actual HTTP API with a temporary artificial checkpoint."""\nimport io\nimport json\nimport os\nimport socket\nimport sys\nimport tempfile\nimport threading\nimport time\nimport unittest\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\nfrom unittest.mock import patch\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \'ml\'))\nos.environ[\'NO_ALBUMENTATIONS_UPDATE\'] = \'1\'\nimport torch\nimport uvicorn\nfrom PIL import Image\nfrom model import create_model, CLASS_NAMES\nfrom app import inference, main\n\nclass APITests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(2)\n        cls.temp = tempfile.TemporaryDirectory()\n        checkpoint=Path(cls.temp.name)/\'test.pt\'\n        torch.save(dict(model_state=create_model(pretrained=False).state_dict(),\n                        class_names=CLASS_NAMES,architecture=\'mobilenet_v3_small\',image_size=64),checkpoint)\n        with patch.object(inference,\'CHECKPOINT\',checkpoint):\n            predictor=inference.Predictor()\n        cls.predictor_patch=patch.object(main,\'predictor\',predictor);cls.predictor_patch.start()\n        cls.sock=socket.socket();cls.sock.bind((\'127.0.0.1\',0))\n        cls.url=f\'http://127.0.0.1:{cls.sock.getsockname()[1]}\'\n        cls.server=uvicorn.Server(uvicorn.Config(main.app,log_level=\'error\'))\n        cls.thread=threading.Thread(target=cls.server.run,kwargs={\'sockets\':[cls.sock]},daemon=True)\n        cls.thread.start()\n        for _ in range(100):\n            if cls.server.started: break\n            time.sleep(.02)\n        if not cls.server.started: raise RuntimeError(\'API did not start\')\n\n    @classmethod\n    def tearDownClass(cls):\n        cls.server.should_exit=True;cls.thread.join(timeout=5)\n        cls.sock.close();cls.predictor_patch.stop();cls.temp.cleanup()\n\n    def upload(self,raw,content_type=\'image/png\'):\n        boundary=\'skinsighttest\'\n        body=(f\'--{boundary}\\r\\nContent-Disposition: form-data; name="image"; filename="image.png"\\r\\nContent-Type: {content_type}\\r\\n\\r\\n\').encode()+raw+f\'\\r\\n--{boundary}--\\r\\n\'.encode()\n        return urllib.request.urlopen(urllib.request.Request(self.url+\'/api/predict\',data=body,headers={\'Content-Type\':f\'multipart/form-data; boundary={boundary}\'}))\n\n    def test_prediction_and_html(self):\n        with urllib.request.urlopen(self.url+\'/\') as response:\n            self.assertIn(b\'MobileNet\',response.read())\n        with urllib.request.urlopen(self.url+\'/api/health\') as response:\n            self.assertTrue(json.load(response)[\'model_ready\'])\n        raw=io.BytesIO();Image.new(\'RGB\',(64,64),\'brown\').save(raw,format=\'PNG\')\n        with self.upload(raw.getvalue()) as response:\n            result=json.load(response)\n            self.assertIn(result[\'label\'],CLASS_NAMES)\n            self.assertIn(\'guidance\',result)\n            self.assertIn(\'stage\',result)\n            self.assertIn(\'not a medical diagnosis\',result[\'medical_disclaimer\'])\n\n    def test_invalid_image_rejected(self):\n        with self.assertRaises(urllib.error.HTTPError) as error: self.upload(b\'invalid\')\n        self.assertEqual(error.exception.code,400)\n\n    def test_untrained_model_returns_503(self):\n        with patch.object(main.predictor,\'model\',None):\n            with self.assertRaises(urllib.error.HTTPError) as error: self.upload(b\'invalid\')\n        self.assertEqual(error.exception.code,503)\n\nif __name__ == \'__main__\': unittest.main()\n', 'tests/test_workflow.py': '"""Pipeline regression tests use artificial images, never medical accuracy claims."""\nimport io\nimport os\nimport sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nsys.path.insert(0, str(ROOT / \'ml\'))\nos.environ[\'NO_ALBUMENTATIONS_UPDATE\'] = \'1\'\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom model import create_model\nfrom train import split_df, evaluate, tfms\nfrom dataset import ManifestDataset, CLASS_NAMES\nfrom prepare_combined import build_pad_manifest, build_isic_manifest\nfrom torch.utils.data import DataLoader\nfrom app import inference\n\ntorch.set_num_threads(2)\n\nclass WorkflowTests(unittest.TestCase):\n    def test_training_checkpoint_loads_and_predicts(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            checkpoint = Path(tmp) / \'model.pt\'\n            model = create_model(pretrained=False, freeze_backbone=False)\n            torch.save(dict(model_state=model.state_dict(), class_names=CLASS_NAMES,\n                            architecture=\'mobilenet_v3_small\', image_size=64,\n                            validation_report={\'value\': 0.2}), checkpoint)\n            with patch.object(inference, \'CHECKPOINT\', checkpoint):\n                predictor = inference.Predictor()\n            raw = io.BytesIO()\n            Image.new(\'RGB\', (64, 64), \'brown\').save(raw, format=\'PNG\')\n            result = predictor.predict_bytes(raw.getvalue())\n            self.assertTrue(predictor.ready)\n            self.assertIn(result[\'label\'], CLASS_NAMES)\n            self.assertEqual(len(result[\'top_predictions\']), 3)\n            self.assertTrue(0 <= result[\'confidence\'] <= 100)\n\n    def test_patient_split_no_overlap(self):\n        rows = [dict(label=CLASS_NAMES[i % 6], source=\'PAD-UFES-20\',\n                     patient_id=f\'PAD::{i // 2}\', original_id=str(i)) for i in range(60)]\n        train, val = split_df(pd.DataFrame(rows))\n        self.assertFalse(set(train.patient_id) & set(val.patient_id))\n        self.assertEqual(len(train) + len(val), 60)\n\n    def test_recursive_manifest_mapping_and_missing_image(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp); nested = root / \'imgs_part_1\'; nested.mkdir()\n            Image.new(\'RGB\', (64,64)).save(nested / \'PAD_1.png\')\n            Image.new(\'RGB\', (64,64)).save(root / \'ISIC_1.jpg\')\n            pd.DataFrame([dict(img_id=\'PAD_1.png\', diagnostic=\'NEV\', patient_id=7)]).to_csv(root/\'pad.csv\',index=False)\n            pd.DataFrame([dict(image=\'ISIC_1\', NV=1, MEL=0, DF=0),dict(image=\'excluded\', NV=0, MEL=0, DF=1)]).to_csv(root/\'isic.csv\',index=False)\n            pad=build_pad_manifest(root/\'pad.csv\',root)\n            isic=build_isic_manifest(root/\'isic.csv\',root)\n            self.assertEqual(pad.iloc[0].patient_id, \'PAD::7\')\n            self.assertEqual(isic.iloc[0].label, \'NEV\')\n            self.assertEqual(len(isic),1)\n            (nested/\'PAD_1.png\').unlink()\n            with self.assertRaises(ValueError): build_pad_manifest(root/\'pad.csv\',root)\n\n    def test_isic_metadata_uses_lesions_and_rejects_dummy_patients(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root=Path(tmp)\n            Image.new(\'RGB\',(64,64)).save(root/\'ISIC_1.jpg\')\n            pd.DataFrame([dict(image=\'ISIC_1\', NV=1)]).to_csv(root/\'gt.csv\',index=False)\n            pd.DataFrame([dict(isic_id=\'ISIC_1\',patient_id=\'dummy_0\',lesion_id=\'lesion_7\')]).to_csv(root/\'meta.csv\',index=False)\n            manifest=build_isic_manifest(root/\'gt.csv\',root,root/\'meta.csv\')\n            self.assertEqual(manifest.iloc[0].patient_id,\'\')\n            self.assertEqual(manifest.iloc[0].lesion_id,\'lesion_7\')\n            self.assertEqual(manifest.iloc[0].grouping_level,\'lesion\')\n            self.assertEqual(len(manifest.iloc[0].image_sha256),64)\n\n    def test_conflicting_isic_labels_are_rejected(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            root=Path(tmp)\n            pd.DataFrame([dict(image=\'ISIC_1\', NV=1, DF=1)]).to_csv(root/\'gt.csv\',index=False)\n            with self.assertRaisesRegex(ValueError,\'one-hot\'):\n                build_isic_manifest(root/\'gt.csv\',root)\n\n    def test_evaluation_handles_absent_classes(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            image=Path(tmp)/\'one.png\'; Image.new(\'RGB\',(64,64)).save(image)\n            ds=ManifestDataset(pd.DataFrame([dict(image_path=str(image),label=\'NEV\')]),tfms(False,64))\n            loss,score,report=evaluate(create_model(pretrained=False),DataLoader(ds,batch_size=1),torch.device(\'cpu\'))\n            self.assertIn(\'MEL\',report)\n            self.assertTrue(loss >= 0)\n\nif __name__ == \'__main__\': unittest.main()\n', 'tests/test_evaluation.py': "import sys\nimport unittest\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'ml'))\nfrom splits import linked_groups, split_three_way\nfrom evaluation import summarize\n\nclass EvaluationTests(unittest.TestCase):\n    def test_transitive_patient_lesion_and_duplicate_grouping(self):\n        frame = pd.DataFrame([\n            dict(source='PAD', original_id='a',patient_id='p1',lesion_id='l1',image_sha256='hashA'),\n            dict(source='PAD', original_id='b',patient_id='p1',lesion_id='l2',image_sha256='hashB'),\n            dict(source='ISIC',original_id='c',patient_id='',lesion_id='l3',image_sha256='hashB'),\n            dict(source='ISIC',original_id='d',patient_id='',lesion_id='l3',image_sha256='hashD'),\n            dict(source='ISIC',original_id='e',patient_id='',lesion_id='',image_sha256='unique'),\n        ])\n        groups = linked_groups(frame)\n        self.assertEqual(len(set(groups[:4])),1)\n        self.assertNotEqual(groups.iloc[0],groups.iloc[4])\n\n    def test_three_partitions_are_disjoint_and_keep_all_strata(self):\n        frame=pd.DataFrame([dict(source=source,label=label,original_id=f'{source}_{label}_{i}',\n                                 patient_id=f'{source}_{label}_{i//2}',image_sha256=f'{source}_{label}_{i}')\n                            for source in ['PAD','ISIC'] for label in ['MEL','NEV'] for i in range(30)])\n        parts=split_three_way(frame)\n        for i,part in enumerate(parts):\n            self.assertEqual(set(part.label),{'MEL','NEV'})\n            self.assertEqual(set(part.source),{'PAD','ISIC'})\n            for other in parts[:i]:\n                self.assertFalse(set(part._group)&set(other._group))\n        self.assertEqual(sum(map(len,parts)),len(frame))\n        again=split_three_way(frame)\n        self.assertEqual(parts[2].original_id.tolist(),again[2].original_id.tolist())\n\n    def test_specificity_sensitivity_auc_and_confusion_matrix(self):\n        probabilities=np.array([[.8,.2,0,0,0,0],[.1,.9,0,0,0,0],[.7,.3,0,0,0,0]])\n        report=summarize([0,1,1],probabilities)\n        self.assertEqual(report['confusion_matrix'][1][0],1)\n        self.assertEqual(report['per_class']['BCC']['sensitivity'],.5)\n        self.assertEqual(report['per_class']['ACK']['specificity'],.5)\n        self.assertIsNone(report['per_class']['MEL']['sensitivity'])\n        self.assertIsNone(report['per_class']['MEL']['roc_auc'])\n        self.assertGreaterEqual(report['expected_calibration_error_10_bins'],0)\n\nif __name__ == '__main__': unittest.main()\n", 'app/static/app.js': 'const input = document.getElementById("fileInput");\nconst chooseBtn = document.getElementById("chooseBtn");\nconst analyzeBtn = document.getElementById("analyzeBtn");\nconst preview = document.getElementById("preview");\nconst placeholder = document.getElementById("placeholder");\nconst fileName = document.getElementById("fileName");\nconst error = document.getElementById("error");\n\nchooseBtn.addEventListener("click", () => input.click());\n\ninput.addEventListener("change", () => {\n  error.hidden = true;\n  const file = input.files?.[0];\n  if (!file) return;\n\n  if (!file.type.startsWith("image/")) {\n    showError("Please select an image file.");\n    return;\n  }\n\n  const url = URL.createObjectURL(file);\n  preview.src = url;\n  preview.hidden = false;\n  placeholder.hidden = true;\n  fileName.textContent = `${file.name} · ${(file.size / 1024 / 1024).toFixed(2)} MB`;\n  analyzeBtn.disabled = false;\n});\n\nanalyzeBtn.addEventListener("click", async () => {\n  const file = input.files?.[0];\n  if (!file) return;\n\n  error.hidden = true;\n  analyzeBtn.disabled = true;\n  analyzeBtn.textContent = "Analyzing…";\n\n  const form = new FormData();\n  form.append("image", file);\n\n  try {\n    const res = await fetch("/api/predict", { method: "POST", body: form });\n    const data = await res.json();\n    if (!res.ok) throw new Error(data.detail || "Analysis failed.");\n    renderResult(data);\n  } catch (e) {\n    showError(e.message);\n  } finally {\n    analyzeBtn.disabled = false;\n    analyzeBtn.textContent = "Analyze Image";\n  }\n});\n\nfunction renderResult(data) {\n  document.getElementById("emptyResult").hidden = true;\n  document.getElementById("result").hidden = false;\n\n  document.getElementById("diagnosis").textContent = data.guidance.name;\n  document.getElementById("confidence").textContent = `${data.confidence.toFixed(1)}%`;\n  document.getElementById("risk").textContent = data.guidance.risk;\n  document.getElementById("message").textContent = data.guidance.message;\n  document.getElementById("doctorText").textContent = data.guidance.urgency;\n  document.getElementById("stageStatus").textContent = data.stage.status;\n  document.getElementById("stageText").textContent = data.stage.explanation;\n  document.getElementById("disclaimer").textContent = data.medical_disclaimer;\n\n  const box = document.getElementById("probabilities");\n  box.innerHTML = "";\n  data.top_predictions.forEach(p => {\n    const row = document.createElement("div");\n    row.className = "prob-row";\n    row.innerHTML = `\n      <div class="prob-label"><span>${p.label}</span><span>${p.probability.toFixed(1)}%</span></div>\n      <div class="track"><div class="fill" style="width:${Math.max(1, p.probability)}%"></div></div>\n    `;\n    box.appendChild(row);\n  });\n}\n\nfunction showError(msg) {\n  error.textContent = msg;\n  error.hidden = false;\n}\n', 'app/static/styles.css': ':root {\n  --bg: #08111f;\n  --panel: #0f1c2f;\n  --panel-2: #13243a;\n  --text: #f5f8ff;\n  --muted: #9eb0c9;\n  --line: rgba(255,255,255,.10);\n  --accent: #42d3a4;\n  --accent-2: #68a7ff;\n  --danger: #ff7a8a;\n  --warning: #ffcc66;\n}\n\n* { box-sizing: border-box; }\nbody {\n  margin: 0;\n  font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;\n  background:\n    radial-gradient(circle at 15% 0%, rgba(104,167,255,.17), transparent 35%),\n    radial-gradient(circle at 90% 10%, rgba(66,211,164,.11), transparent 25%),\n    var(--bg);\n  color: var(--text);\n  min-height: 100vh;\n}\n\n.topbar {\n  display: flex;\n  justify-content: space-between;\n  align-items: center;\n  padding: 22px clamp(20px, 5vw, 72px);\n  border-bottom: 1px solid var(--line);\n  backdrop-filter: blur(12px);\n  position: sticky;\n  top: 0;\n  background: rgba(8,17,31,.82);\n  z-index: 5;\n}\n\n.brand { font-size: 1.2rem; font-weight: 800; }\n.brand span { color: var(--accent); }\n.badge {\n  padding: 7px 11px;\n  border: 1px solid var(--line);\n  border-radius: 999px;\n  color: var(--muted);\n  font-size: .78rem;\n}\n\n.page { width: min(1180px, calc(100% - 36px)); margin: 0 auto; padding-bottom: 60px; }\n\n.hero {\n  display: grid;\n  grid-template-columns: 1.5fr .8fr;\n  gap: 30px;\n  padding: 56px 0 34px;\n  align-items: end;\n}\n\n.eyebrow {\n  color: var(--accent);\n  font-weight: 800;\n  letter-spacing: .14em;\n  font-size: .72rem;\n  margin-bottom: 10px;\n}\n\nh1 { font-size: clamp(2.2rem, 5vw, 4.4rem); line-height: .98; margin: 0; max-width: 850px; }\nh2 { font-size: 1.25rem; margin-top: 0; }\nh3 { font-size: 1.8rem; margin: 3px 0 0; }\n.lead { color: var(--muted); max-width: 760px; font-size: 1.05rem; line-height: 1.7; }\n\n.feature-row { display: flex; flex-wrap: wrap; gap: 10px; margin-top: 22px; }\n.feature-row span {\n  padding: 8px 11px;\n  border-radius: 10px;\n  background: rgba(255,255,255,.055);\n  border: 1px solid var(--line);\n  color: #dce8fb;\n  font-size: .86rem;\n}\n\n.status-card {\n  display: flex;\n  gap: 13px;\n  align-items: center;\n  background: linear-gradient(145deg, rgba(255,255,255,.07), rgba(255,255,255,.03));\n  border: 1px solid var(--line);\n  border-radius: 18px;\n  padding: 18px;\n}\n.status-card p { color: var(--muted); margin: 5px 0 0; font-size: .82rem; }\n.status-dot { width: 13px; height: 13px; border-radius: 50%; flex: 0 0 auto; }\n.status-dot.ready { background: var(--accent); box-shadow: 0 0 18px var(--accent); }\n.status-dot.not-ready { background: var(--warning); }\n\n.grid { display: grid; grid-template-columns: 1fr 1fr; gap: 22px; }\n.card {\n  background: linear-gradient(145deg, rgba(20,39,62,.96), rgba(13,28,47,.96));\n  border: 1px solid var(--line);\n  border-radius: 22px;\n  padding: clamp(18px, 3vw, 30px);\n  box-shadow: 0 24px 80px rgba(0,0,0,.18);\n}\n\n.muted { color: var(--muted); }\n.small { font-size: .8rem; }\n\n.dropzone {\n  height: 330px;\n  border: 1px dashed rgba(255,255,255,.22);\n  background: rgba(3,10,20,.25);\n  border-radius: 18px;\n  display: grid;\n  place-items: center;\n  overflow: hidden;\n  margin: 22px 0 14px;\n  text-align: center;\n}\n.dropzone img { width: 100%; height: 100%; object-fit: cover; }\n.camera-icon { font-size: 3rem; margin-bottom: 8px; }\n\n.actions { display: flex; gap: 10px; flex-wrap: wrap; }\n.btn {\n  border: 0;\n  border-radius: 12px;\n  padding: 12px 16px;\n  font-weight: 800;\n  cursor: pointer;\n}\n.btn.primary { background: var(--accent); color: #062016; }\n.btn.secondary { background: #233b5b; color: white; }\n.btn:disabled { opacity: .48; cursor: not-allowed; }\n\n.empty-state {\n  min-height: 440px;\n  display: grid;\n  place-items: center;\n  align-content: center;\n  color: var(--muted);\n}\n.pulse-orb {\n  width: 70px; height: 70px; border-radius: 50%;\n  background: radial-gradient(circle at 35% 30%, #8ec2ff, #315f99);\n  box-shadow: 0 0 70px rgba(104,167,255,.35);\n  margin-bottom: 12px;\n}\n\n.result-heading { display: flex; justify-content: space-between; align-items: center; gap: 15px; }\n.confidence { text-align: right; }\n.confidence span { font-size: 1.65rem; font-weight: 900; color: var(--accent); display: block; }\n.confidence small { color: var(--muted); }\n\n.probabilities { margin: 25px 0; }\n.prob-row { margin: 13px 0; }\n.prob-label { display: flex; justify-content: space-between; font-size: .83rem; margin-bottom: 6px; }\n.track { height: 8px; border-radius: 999px; background: rgba(255,255,255,.08); overflow: hidden; }\n.fill { height: 100%; border-radius: inherit; background: linear-gradient(90deg, var(--accent-2), var(--accent)); }\n\n.info-box {\n  padding: 15px;\n  border-radius: 14px;\n  background: rgba(255,255,255,.05);\n  border: 1px solid var(--line);\n  margin: 11px 0;\n}\n.info-box p { color: #ced9ea; line-height: 1.55; margin-bottom: 0; }\n.info-box.doctor { border-color: rgba(66,211,164,.24); }\n.info-box.stage { border-color: rgba(255,204,102,.24); }\n\n.disclaimer {\n  color: var(--muted);\n  line-height: 1.5;\n  font-size: .79rem;\n  margin-top: 18px;\n}\n.error {\n  background: rgba(255,122,138,.12);\n  border: 1px solid rgba(255,122,138,.35);\n  color: #ffc0c8;\n  padding: 10px 12px;\n  border-radius: 10px;\n}\n\n.safety {\n  margin-top: 22px;\n  padding: 28px;\n  border-radius: 22px;\n  border: 1px solid var(--line);\n  background: rgba(255,255,255,.025);\n}\n.safety-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }\n.safety-grid div {\n  display: flex; gap: 10px;\n  padding: 14px;\n  background: rgba(255,255,255,.04);\n  border-radius: 12px;\n  color: #dbe6f5;\n  line-height: 1.4;\n}\n.safety-grid b { color: var(--accent); }\n\n@media (max-width: 820px) {\n  .hero, .grid { grid-template-columns: 1fr; }\n  .hero { padding-top: 32px; }\n  .safety-grid { grid-template-columns: 1fr; }\n  .dropzone { height: 300px; }\n  .badge { display: none; }\n}\n', 'app/templates/index.html': '<!doctype html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <title>SkinSight AI</title>\n  <link rel="stylesheet" href="/static/styles.css">\n</head>\n<body>\n  <header class="topbar">\n    <div class="brand">SkinSight <span>AI</span></div>\n    <div class="badge">Research / Educational Prototype</div>\n  </header>\n\n  <main class="page">\n    <section class="hero">\n      <div>\n        <p class="eyebrow">SMARTPHONE SKIN-LESION SCREENING</p>\n        <h1>Capture. Analyze. Understand the next step.</h1>\n        <p class="lead">\n          A new own-trained MobileNetV3-based skin-lesion classifier designed around\n          smartphone clinical images.\n        </p>\n        <div class="feature-row">\n          <span>📷 Mobile camera</span>\n          <span>🧠 Own-trained model</span>\n          <span>🩺 Doctor guidance</span>\n          <span>🔬 Stage-safe explanation</span>\n        </div>\n      </div>\n      <div class="status-card">\n        <div class="status-dot {{ \'ready\' if model_ready else \'not-ready\' }}"></div>\n        <div>\n          <strong>{{ \'Model ready\' if model_ready else \'Model not trained yet\' }}</strong>\n          <p>\n            {% if model_ready %}\n              {{ model_meta.get(\'architecture\', \'MobileNetV3-Small\') }} · {{ model_meta.get(\'dataset\', \'PAD-UFES-20\') }}\n            {% else %}\n              The website is ready. Train the model to enable predictions.\n            {% endif %}\n          </p>\n        </div>\n      </div>\n    </section>\n\n    <section class="grid">\n      <div class="card capture-card">\n        <h2>1. Take or upload a skin photo</h2>\n        <p class="muted">Use natural light, keep the lesion in focus, and avoid heavy filters.</p>\n\n        <div id="dropzone" class="dropzone">\n          <img id="preview" alt="Selected skin image preview" hidden>\n          <div id="placeholder">\n            <div class="camera-icon">📷</div>\n            <p>Select an image or use your phone camera</p>\n          </div>\n        </div>\n\n        <input id="fileInput" type="file" accept="image/*" capture="environment" hidden>\n\n        <div class="actions">\n          <button id="chooseBtn" class="btn primary">Take Photo / Choose Image</button>\n          <button id="analyzeBtn" class="btn secondary" disabled>Analyze Image</button>\n        </div>\n\n        <p id="fileName" class="small muted"></p>\n        <p id="error" class="error" hidden></p>\n      </div>\n\n      <div class="card result-card">\n        <h2>2. AI screening result</h2>\n        <div id="emptyResult" class="empty-state">\n          <div class="pulse-orb"></div>\n          <p>Your analysis will appear here.</p>\n        </div>\n\n        <div id="result" hidden>\n          <div class="result-heading">\n            <div>\n              <p class="eyebrow">TOP CLASS</p>\n              <h3 id="diagnosis">—</h3>\n            </div>\n            <div class="confidence">\n              <span id="confidence">0%</span>\n              <small>model score</small>\n            </div>\n          </div>\n\n          <div id="probabilities" class="probabilities"></div>\n\n          <div class="info-box">\n            <strong id="risk">—</strong>\n            <p id="message"></p>\n          </div>\n\n          <div class="info-box doctor">\n            <strong>Doctor recommendation</strong>\n            <p id="doctorText"></p>\n          </div>\n\n          <div class="info-box stage">\n            <strong>Stage information</strong>\n            <p><b id="stageStatus"></b></p>\n            <p id="stageText"></p>\n          </div>\n\n          <p class="disclaimer" id="disclaimer"></p>\n        </div>\n      </div>\n    </section>\n\n    <section class="safety">\n      <h2>What this project does — and does not do</h2>\n      <div class="safety-grid">\n        <div><b>✓</b><span>Classifies six PAD-UFES-20 lesion categories.</span></div>\n        <div><b>✓</b><span>Works with uploaded or mobile-camera photos.</span></div>\n        <div><b>✓</b><span>Shows probabilities and next-step guidance.</span></div>\n        <div><b>✕</b><span>Does not claim a biopsy diagnosis.</span></div>\n        <div><b>✕</b><span>Does not invent Stage I–IV from a photograph.</span></div>\n        <div><b>✓</b><span>Explains when professional assessment is needed.</span></div>\n      </div>\n    </section>\n  </main>\n\n  <script src="/static/app.js"></script>\n</body>\n</html>\n'}
for name, contents in FILES.items():
    destination = PROJECT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents, encoding='utf-8')
print('Prepared corrected project:', PROJECT)


In [ ]:
import kagglehub
pad = Path(kagglehub.dataset_download('mahdavi1202/skin-cancer'))
isic = Path(kagglehub.dataset_download('nischaydnk/isic-2019-jpg-224x224-resized'))
gt_download = Path(kagglehub.dataset_download('andrewmvd/isic-2019', path='ISIC_2019_Training_GroundTruth.csv'))
import urllib.request
isic_meta = PROJECT / 'data/ISIC_2019_Training_Metadata.csv'
isic_meta.parent.mkdir(parents=True, exist_ok=True)
with urllib.request.urlopen('https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Metadata.csv', timeout=120) as response:
    isic_meta.write_bytes(response.read())
def locate(root, name):
    if root.is_file() and root.name == name:
        return root
    matches = list(root.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {name} in {root}; found {len(matches)}')
    return matches[0]
pad_meta = locate(pad, 'metadata.csv')
isic_gt = locate(gt_download, 'ISIC_2019_Training_GroundTruth.csv')
print('PAD:',pad, 'ISIC:',isic, 'ground truth:',isic_gt)


In [ ]:
import pandas as pd
manifest = PROJECT / 'data/combined_manifest.csv'
subprocess.run([sys.executable, 'ml/prepare_combined.py',
    '--pad-meta', str(pad_meta), '--pad-root', str(pad),
    '--isic-gt', str(isic_gt), '--isic-root', str(isic),
    '--isic-meta', str(isic_meta),
    '--out', str(manifest)], cwd=PROJECT, check=True)
frame = pd.read_csv(manifest)
counts = frame.groupby('source').size().to_dict()
assert counts == {'PAD-UFES-20': 2298, 'ISIC-2019': 24839}, counts
print('Verified all 27,137 supported records matched images.')


In [ ]:
# Baseline training; ten epochs do not guarantee a clinically useful model.
subprocess.run([sys.executable, 'ml/train.py', '--manifest', str(manifest),
    '--epochs', '10', '--batch-size', '32', '--workers', '2', '--size', '160',
    '--fine-tune', '--device', 'cuda', '--output-dir', str(PROJECT / 'models')],
    cwd=PROJECT, check=True)


In [ ]:
import json, shutil
print('Validation:', (PROJECT/'models/metrics.json').read_text())
print('Held-out test:', (PROJECT/'models/test_metrics.json').read_text())
print('Grouping coverage:', (PROJECT/'models/split_summary.json').read_text())
# Check loading and prediction mechanics against a held-out image.
# This is software validation, not an independent accuracy estimate.
sys.path.insert(0,str(PROJECT))
os.environ['SKINSIGHT_MODEL_PATH'] = str(PROJECT/'models/skinsight_mobilenetv3_small.pt')
from app.inference import Predictor
predictor = Predictor()
assert predictor.ready
held_out = pd.read_csv(PROJECT/'models/validation_split.csv')
print('Functional prediction:', predictor.predict_bytes(Path(held_out.iloc[0].image_path).read_bytes()))
# Include app and trained checkpoint, omit image paths/manifests referring to Kaggle.
export = Path('/kaggle/working/skinsight_export')
if export.exists():
    raise RuntimeError('Export directory exists; choose a new path to preserve previous results.')
shutil.copytree(PROJECT, export/'SkinSight_AI', ignore=shutil.ignore_patterns('data','__pycache__','*_split.csv'))
archive = shutil.make_archive('/kaggle/working/skinsight_trained_app', 'zip', export)
print('Download this file from notebook output:', archive)


Download `skinsight_trained_app.zip` from the notebook output. Unzip on Windows, open PowerShell in its `SkinSight_AI` folder, and follow README installation/run steps. Metrics include source-specific validation and a separate held-out test. Patient independence depends on grouping coverage. Do not tune using the held-out test or use this model for clinical decisions.